# Task 4.2 — Predict HAI Magnitude for H3N2 A/Massachusetts/18/2022 (D28)

**4.2 predict magnitude of antibody response - H3N2 A/Massachusetts/18/2022 (D28)**
* Training Data: Demographics + Day 0 + Day 7 innate
* Assay: HAI / Measure: Single strain titer / Metric: Spearman correlation
* Full description: HAI titer for H3N2 A/Massachusetts/18/2022 at Day 28

---

## Design notes

**Proxy target:** The target strain (H3N2 A/Massachusetts/18/2022) is absent from the training data.
We approximate it by averaging the Day 28 HAI titers of all other H3N2 strains present in the dataset.
All HAI columns are then dropped to prevent leakage.

**Spearman correlation:** ranks predictions and truth; rewards monotonic agreement regardless of scale.
Robust to outliers. Score: 1.0 = perfect, 0.0 = no signal, -1.0 = reversed.

**5-fold cross-validation:** each participant's prediction is made by a model that never saw them during training.

In [1]:
!pip install h2o

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.4/266.4 MB 3.8 MB/s eta 0:00:00


In [2]:
TARGET_COL = 'H3N2_proxy_d28'  # proxy: target strain absent from training data
AUTO_ML_MAX_RUNTIME_SECONDS = 600

In [3]:
PARQUET_PATH = '../merged_data/combined.parquet'
CHALLENGE_DATA_PATH = '../cleaned_data'
SUBMISSION_PATH = '../automl_submission'

In [4]:
import io
import os
import warnings
from contextlib import redirect_stderr, redirect_stdout

import h2o
import numpy as np
import pandas as pd
from h2o.automl import H2OAutoML
from scipy.stats import spearmanr

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Once your Drive is mounted, you'll need to specify the correct path to your data. If your `merged_data` folder is directly in your Google Drive, the path would likely be something like `/content/drive/MyDrive/merged_data/`.

Let's update the `PARQUET_PATH` and `CHALLENGE_DATA_PATH` variables to point to the correct location within your Google Drive.

In [6]:
# Update these paths to reflect the actual location in your Google Drive
# For example, if 'merged_data' and 'cleaned_data' are directly in 'MyDrive':
# DRIVE_BASE_PATH = '/content/drive/MyDrive'
# PARQUET_PATH = f'{DRIVE_BASE_PATH}/merged_data/combined.parquet'
# CHALLENGE_DATA_PATH = f'{DRIVE_BASE_PATH}/cleaned_data'

# Or if they are in a subfolder like 'MyProject':
# DRIVE_BASE_PATH = '/content/drive/MyDrive/MyProject'
# PARQUET_PATH = f'{DRIVE_BASE_PATH}/merged_data/combined.parquet'
# CHALLENGE_DATA_PATH = f'{DRIVE_BASE_PATH}/cleaned_data'

# Please replace the example path with your actual path after mounting your drive.
# For now, I'm providing a placeholder that you need to adjust.
DRIVE_BASE_PATH = '/content/drive/MyDrive/cmi-flu-prediction-challenge-capstone' # <<<-- **EDITED**
PARQUET_PATH = f'{DRIVE_BASE_PATH}/merged_data/combined.parquet'
CHALLENGE_DATA_PATH = f'{DRIVE_BASE_PATH}/cleaned_data'

print(f"Updated PARQUET_PATH: {PARQUET_PATH}")
print(f"Updated CHALLENGE_DATA_PATH: {CHALLENGE_DATA_PATH}")

Updated PARQUET_PATH: /content/drive/MyDrive/cmi-flu-prediction-challenge-capstone/merged_data/combined.parquet
Updated CHALLENGE_DATA_PATH: /content/drive/MyDrive/cmi-flu-prediction-challenge-capstone/cleaned_data


In [7]:
challenge_participants = pd.read_csv(CHALLENGE_DATA_PATH + '/challenge_participants_cleaned.csv')
challenge_hai = pd.read_csv(CHALLENGE_DATA_PATH + '/challenge_hai_cleaned.csv')
challenge_data = challenge_hai.merge(challenge_participants, on='participant_id', how='inner')
print(f'Challenge shape: {challenge_data.shape}')

Challenge shape: (40, 23)


### Preprocessing — proxy target\n\nThe target strain is absent from the training set. We create a proxy by averaging all H3N2 Day 28 HAI titers.\nRows are filtered to those with a valid proxy target, then all-null columns are removed on the filtered subset.\nThe `_d28`/`_d365` columns are excluded from features at training time to prevent leakage.

In [8]:
df = pd.read_parquet(PARQUET_PATH)
print(f'Raw shape: {df.shape}')

Raw shape: (3757, 110063)


In [9]:
# Create proxy target: row-wise average of all H3N2 d28 HAI strains
h3n2_d28_cols = [c for c in df.columns if c.startswith('HAI_') and 'H3N2' in c and c.endswith('_d28')]
df[TARGET_COL] = df[h3n2_d28_cols].mean(axis=1)
print(f'Proxy target averaged from {len(h3n2_d28_cols)} H3N2 d28 strains')

Proxy target averaged from 25 H3N2 d28 strains


In [ ]:
# Filter to rows with a valid proxy target first
df = df[df[TARGET_COL].notna()].reset_index(drop=True)
print(f'Rows with valid proxy target: {len(df)}')

# Drop all-null columns on the filtered subset
all_null_cols = df.columns[df.isna().all()].tolist()
df = df.drop(columns=all_null_cols)
print(f'Dropped {len(all_null_cols)} all-null columns → {df.shape[1]} remaining')

# Drop columns with 85%+ missing values
before = df.shape[1]
df = df.loc[:, df.notna().mean() >= 0.15]
print(f'Dropped {before - df.shape[1]} columns with ≥85% missing → {df.shape[1]} remaining')

In [11]:
print(f'Shape: {df.shape}')
print(f'Target stats:\n{df[TARGET_COL].describe()}')
print(f'\ndtype counts:\n{df.dtypes.value_counts()}')
print(f'\nMissing per column (top 10):\n{df.isna().sum().sort_values(ascending=False).head(10)}')

Shape: (3568, 83554)
Target stats:
count    3568.000000
mean        5.860525
std         2.024357
min         0.000000
25%         4.321928
50%         5.946928
75%         7.321928
max        14.321928
Name: H3N2_proxy_d28, dtype: float64

dtype counts:
float64    83549
object         5
Name: count, dtype: int64

Missing per column (top 10):
FLOW_Classical monocytes_d7              3567
FLOW_Neutrophils + Basophils_d0          3567
FLOW_Classical monocytes_d0              3567
FLOW_Neutrophils + Basophils_d7          3567
FLOW_Conventional DC_d0                  3567
FLOW_Conventional DC_d7                  3567
FLOW_neutrophils_d0                      3562
FLOW_neutrophils_d7                      3562
FLOW_Neutrophils_d7                      3556
FLOW_Memory class switched B cells_d0    3555
dtype: int64


---
## AutoML Setup

In [12]:
warnings.filterwarnings('ignore', category=UserWarning, module='h2o')
h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.18" 2026-01-20; OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1); OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.12/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmprkyfgr_5
  JVM stdout: /tmp/tmprkyfgr_5/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmprkyfgr_5/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,01 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,1 month and 25 days
H2O_cluster_name:,H2O_from_python_unknownUser_mu8ybi
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,12.75 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


In [ ]:
print(f'Converting to H2O: {df.shape[0]} rows × {df.shape[1]} columns')
data = h2o.H2OFrame(df)
print(f'H2OFrame shape: {data.shape}')

---
## AutoML Training

In [ ]:
# Features: everything available at d0/d7 — exclude all d28/d365 targets and participant_id
x = [c for c in data.columns
     if not c.endswith('_d28') and not c.endswith('_d365')
     and c != 'participant_id']
y = TARGET_COL

train = data[data[y].isna() == 0]
print(f'Training samples: {train.nrows}  |  Features: {len(x)}')

aml = H2OAutoML(max_models=10, seed=1, nfolds=5,
                keep_cross_validation_predictions=True,
                max_runtime_secs=AUTO_ML_MAX_RUNTIME_SECONDS)

_buf = io.StringIO()
with redirect_stdout(_buf), redirect_stderr(_buf):
    aml.train(x=x, y=y, training_frame=train)
print('Training complete.')

In [ ]:
lb = aml.leaderboard
print(lb.head(rows=lb.nrows))

In [ ]:
# Stacked Ensembles don't store CV predictions — fall back to best base model if needed
model = aml.leader
if model.cross_validation_holdout_predictions() is None:
    for m_id in aml.leaderboard['model_id'].as_data_frame()['model_id']:
        m = h2o.get_model(m_id)
        if m.cross_validation_holdout_predictions() is not None:
            model = m
            break

cv_preds = model.cross_validation_holdout_predictions().as_data_frame()['predict']
actuals = train[y].as_data_frame()[y]
rho, pval = spearmanr(actuals, cv_preds)
print(f'Task 4.2 — Spearman (5-fold CV): {rho:.3f}  (p={pval:.4f})')
print(f'Model used for eval: {model.model_id}')

In [ ]:
print(f'Leader model: {aml.leader.model_id}')
print(f'Varimp model: {model.model_id}')
varimp = model.varimp(use_pandas=True)
display(varimp.head(20))
model.varimp_plot(num_of_features=20)

In [ ]:
challenge_hf = h2o.H2OFrame(challenge_data)

# Align column types with training frame to avoid type mismatch errors
for col in challenge_hf.columns:
    if col in data.columns:
        train_type = data[col].types[col]
        test_type = challenge_hf[col].types[col]
        if train_type != test_type:
            if train_type == 'enum':
                challenge_hf[col] = challenge_hf[col].ascharacter().asfactor()
            else:
                challenge_hf[col] = challenge_hf[col].asnumeric()

y_pred = aml.leader.predict(challenge_hf).as_data_frame()['predict']

os.makedirs(SUBMISSION_PATH, exist_ok=True)
results = pd.DataFrame({
    'Participant_ID': challenge_data['participant_id'].values,
    'Task_4.2': np.exp2(y_pred),
})
results.to_csv(f'{SUBMISSION_PATH}/task_4_2.csv', index=False)
results

In [ ]:
h2o.cluster().shutdown()

---
## Conclusion

- **Leader model:** (fill after run)
- **CV Spearman:** (fill after run)

**Target:** Proxy HAI titer for H3N2 A/Massachusetts/18/2022 at D28 (averaged from all available H3N2 d28 strains).
The target strain is absent from the training data; this proxy is our best approximation.

Submission saved to `automl_submission/task_4_2.csv` (raw titer scale via `np.exp2`).